# Vectors

## What's covered

- Two views of a vector — geometric arrow vs algebraic list of numbers
- Vector arithmetic — addition, scalar multiplication, linear combinations
- The dot product (inner product) — algebraic formula and geometric meaning
- Norms — L1, L2, L∞ and why ML uses each one
- Distance and cosine similarity — measuring relationships between vectors
- Projections — splitting a vector into a parallel piece and an orthogonal piece

The vector is the atom of linear algebra. Every dataset, every embedding, every gradient in machine learning is a vector. This notebook builds the operations you need every day.

We use one canonical pair throughout:

$$
\mathbf{u} = \begin{bmatrix} 3 \\ 4 \end{bmatrix}, \qquad
\mathbf{v} = \begin{bmatrix} 4 \\ 3 \end{bmatrix}
$$

Both have a familiar length of 5 (the 3-4-5 triangle), which keeps the arithmetic clean and lets us focus on the ideas.


## Two views of a vector

A vector lives in two heads at once. You will constantly switch between them.

**Geometric view — an arrow.** A vector is an arrow from the origin to a point in space. It has a *direction* and a *length* (also called magnitude). The arrow `(3, 4)` points up and to the right and has length 5.

**Algebraic view — a list of numbers.** A vector is an ordered list of `n` numbers: `[3, 4]`, or `[0.12, -0.85, 1.34, ...]` for an embedding. The number of entries is the **dimension**.

Both views describe the same object. Geometry gives you intuition; algebra gives you a way to compute. In ML you almost always live in the algebraic view because vectors have hundreds or thousands of dimensions — but the geometric picture (in 2D) is what keeps the math honest.

A few conventions used throughout this repo:

- Vectors are column vectors by default. Writing `[3, 4]` means the column `[3; 4]`.
- Bold lowercase in prose (**u**, **v**); italic lowercase for scalars (*a*).
- `u_i` denotes the i-th component of vector **u** (`u_1 = 3`, `u_2 = 4`).


In [ ]:
import numpy as np

u = np.array([3, 4])
v = np.array([4, 3])

print(f"u = {u}, shape = {u.shape}, dim = {u.shape[0]}")
print(f"v = {v}, shape = {v.shape}, dim = {v.shape[0]}")


## Vector addition and scalar multiplication

Two operations, and from them everything else.

**Addition.** Add componentwise:

$$
\mathbf{u} + \mathbf{v} = \begin{bmatrix} u_1 + v_1 \\ u_2 + v_2 \end{bmatrix}
= \begin{bmatrix} 3 + 4 \\ 4 + 3 \end{bmatrix} = \begin{bmatrix} 7 \\ 7 \end{bmatrix}
$$

Geometrically: place **v**'s tail at **u**'s head; the sum is the arrow from origin to the new tip. The "parallelogram rule."

**Scalar multiplication.** Stretch (or flip) by a number:

$$
c \mathbf{u} = \begin{bmatrix} c \, u_1 \\ c \, u_2 \end{bmatrix}
$$

Geometrically: same direction, scaled by `c`. If `c < 0`, the arrow flips to the opposite direction. If `c = 0`, you collapse to the origin.

These two operations are what makes a set of vectors a **vector space** (next notebook). When you combine them — `c_1 u + c_2 v` — you get a **linear combination**, the single most reused phrase in linear algebra.


In [ ]:
print("u + v       =", u + v)
print("u - v       =", u - v)
print("2 * u       =", 2 * u)
print("-1 * u      =", -1 * u)
print("0.5*u + 2*v =", 0.5 * u + 2 * v)  # a linear combination


## The dot product

The dot product (or **inner product**) takes two vectors and returns a single number:

$$
\mathbf{u} \cdot \mathbf{v} = u_1 v_1 + u_2 v_2 + \dots + u_n v_n = \sum_{i=1}^{n} u_i v_i
$$

For our pair: `u · v = 3·4 + 4·3 = 24`.

That's the algebraic definition. The geometric one is what gives the dot product its power:

$$
\mathbf{u} \cdot \mathbf{v} = \|\mathbf{u}\| \, \|\mathbf{v}\| \, \cos\theta
$$

where `||u||` and `||v||` are the lengths and `θ` is the angle between them. So the dot product encodes **both** how long the vectors are and how aligned they are. Three things to memorize:

- `u · v > 0` — vectors point in roughly the same direction (`θ < 90°`).
- `u · v = 0` — vectors are **orthogonal** (perpendicular).
- `u · v < 0` — vectors point in roughly opposite directions (`θ > 90°`).

Orthogonality (`u · v = 0`) is the cleanest, most useful relationship in all of linear algebra; we'll spend an entire notebook on it.


In [ ]:
# Three equivalent ways to compute the dot product
print("u . v (manual)    =", u[0]*v[0] + u[1]*v[1])
print("u . v (np.dot)    =", np.dot(u, v))
print("u . v (@ operator)=", u @ v)


## Norms — measuring length

A **norm** turns a vector into a number that represents its size. The most familiar is what we mean by "length" in everyday space.

**L2 norm (Euclidean).** The straight-line length, from Pythagoras:

$$
\|\mathbf{u}\|_2 = \sqrt{u_1^2 + u_2^2 + \dots + u_n^2} = \sqrt{\mathbf{u} \cdot \mathbf{u}}
$$

For `u = [3, 4]`: `||u||_2 = √(9 + 16) = √25 = 5`.

**L1 norm (Manhattan / taxicab).** Sum of absolute values:

$$
\|\mathbf{u}\|_1 = |u_1| + |u_2| + \dots + |u_n|
$$

For `u = [3, 4]`: `||u||_1 = 7`. Picture a taxi driving on a grid; it cannot cut diagonals.

**L∞ norm (max).** The largest component in absolute value:

$$
\|\mathbf{u}\|_\infty = \max_i |u_i|
$$

For `u = [3, 4]`: `||u||_∞ = 4`.

**Lp norm (general).** All three above are special cases of:

$$
\|\mathbf{u}\|_p = \left( \sum_i |u_i|^p \right)^{1/p}
$$

Why three? They reward different things, and ML uses all of them:

- **L2** is smooth and differentiable → friendly to gradient descent → default for regression losses and ridge regularization.
- **L1** is non-smooth at zero but encourages **sparsity** (many components exactly zero) → lasso regularization, feature selection.
- **L∞** caps the worst component → used in adversarial robustness ("the perturbation cannot move any pixel by more than ε").


In [ ]:
print("L2 norm of u =", np.linalg.norm(u))            # default is L2
print("L1 norm of u =", np.linalg.norm(u, ord=1))
print("L-inf norm of u =", np.linalg.norm(u, ord=np.inf))

# Unit vector — same direction, length 1
u_hat = u / np.linalg.norm(u)
print("u_hat        =", u_hat, "(length =", np.linalg.norm(u_hat), ")")


## Distance and cosine similarity

Norms apply to a single vector. We almost always care about *pairs* of vectors — how far apart, or how similar.

**Distance.** The distance between **u** and **v** is the norm of their difference:

$$
d(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|_2
$$

For our pair: `u - v = [-1, 1]`, so `d(u, v) = √2 ≈ 1.41`.

**Cosine similarity.** From the geometric definition of the dot product, rearrange for the angle:

$$
\cos\theta = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \, \|\mathbf{v}\|}
$$

This is the single most-used similarity metric in machine learning. It is the dot product *normalized by the lengths*, so it ignores magnitude and only measures alignment. Output range: `[-1, 1]`.

- `cos θ = 1` — identical direction
- `cos θ = 0` — orthogonal
- `cos θ = -1` — opposite direction

For our pair: `cos θ = 24 / (5 · 5) = 0.96`, so the angle is about 16°.

**Why ML prefers cosine over Euclidean.** Embeddings often live on or near a sphere of fixed length, where what matters is direction, not magnitude. Two word embeddings can be similar in meaning even if one has slightly larger magnitude. Cosine throws away the magnitude difference; Euclidean does not.


In [ ]:
# Euclidean distance
dist = np.linalg.norm(u - v)
print(f"distance(u, v) = {dist:.4f}")

# Cosine similarity
cos_sim = (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v))
angle_rad = np.arccos(cos_sim)
angle_deg = np.degrees(angle_rad)
print(f"cos(theta)     = {cos_sim:.4f}")
print(f"angle          = {angle_deg:.2f} degrees")


## Projections — splitting a vector into pieces

A projection answers: *"How much of vector **u** points along the direction of vector **v**?"*

Picture shining a light perpendicular to **v** and recording where **u**'s shadow lands on the line through **v**. That shadow is the **projection of u onto v**:

$$
\text{proj}_{\mathbf{v}}(\mathbf{u}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\mathbf{v} \cdot \mathbf{v}} \, \mathbf{v}
$$

The scalar `(u · v) / (v · v)` says *how many copies of **v*** fit into **u** along **v**'s direction. Multiplying by **v** then gives the projected vector.

For our pair: `(u · v) / (v · v) = 24 / 25 = 0.96`, so `proj_v(u) = 0.96 · [4, 3] = [3.84, 2.88]`.

The remaining piece — what's left of **u** after subtracting its projection — is **orthogonal** to **v**:

$$
\mathbf{u}_{\perp} = \mathbf{u} - \text{proj}_{\mathbf{v}}(\mathbf{u})
$$

This is the move at the heart of **linear regression**: when there is no exact solution to `Aw = y`, you project `y` onto the column space of `A` and solve for `w` against the projection instead. Least squares = projection.


In [ ]:
# Project u onto v
scalar = (u @ v) / (v @ v)
proj = scalar * v
print(f"scalar factor       = {scalar:.4f}")
print(f"proj_v(u)           = {proj}")

# The leftover piece, orthogonal to v
perp = u - proj
print(f"u - proj_v(u)       = {perp}")
print(f"(u - proj_v(u)) . v = {perp @ v:.6f}   <- approximately zero, confirming orthogonality")


## Where this appears in ML

Almost everything below is a thin layer on top of the operations above.

- **Embeddings.** Words, images, users, products — represented as vectors. Similarity between two embeddings is **cosine similarity**. Search, recommendation, retrieval-augmented generation: all dot products.
- **k-Nearest Neighbors.** Classify a new point by the labels of its k closest training points, where "closest" is **Euclidean (L2) distance**.
- **Linear regression / single neuron.** `y_hat = w · x + b` — a **dot product** plus a bias.
- **Attention (transformers).** `attention(Q, K) = softmax((Q · K^T) / √d)` — a giant grid of dot products between queries and keys.
- **Normalization layers (LayerNorm, L2 normalization).** Rescale vectors so `||x||_2 = 1`. This is exactly the **unit vector** trick from the norms section.
- **L1 vs L2 regularization.** Ridge adds `λ ||w||_2^2`, lasso adds `λ ||w||_1`. Different norms, different inductive biases — L1 produces **sparse** weights, L2 produces **small** ones.
- **Least squares regression.** Find the `w` whose prediction is the **projection** of `y` onto the column space of `X`.

Next notebook: **vector spaces** — once you have vectors, addition, and scaling, what does the full set of "reachable" vectors look like? That set is a vector space, and span / basis / dimension are its three vocabulary words.
